# MITRE ATT&CK Knowledge Graph — v2.0
**Full pipeline:** Tactics → Techniques → Sub-techniques → Groups → Software → CAPEC → CWE → CVE
**CVE Source:** CISA KEV — highest priority, all actively exploited


## Graph schema
```
(CVE)-[:CAUSE_WEAKNESS]->(CWE)
(CWE)<-[:TARGETS_WEAKNESS]-(CAPEC)
(CAPEC)-[:MAPS_TO_TECHNIQUE]->(Technique)
(Technique)-[:BELONGS_TO]->(Tactic)
(Technique)-[:HAS_SUBTECHNIQUE]->(Technique)   # sub-techniques
(Group)-[:USES]->(Technique)
(Group)-[:USES]->(Software)
(Software)-[:USES]->(Technique)
(CAPEC)-[:CHILD_OF]->(CAPEC)                   # CAPEC hierarchy
```

## Run order
1. Cell 1 – Constraints & indexes (run once)
2. Cell 2 – Ingest ATT&CK (Tactics + Techniques + Sub-techniques + Groups + Software)
3. Cell 3 – Ingest CAPEC nodes + hierarchy + CWE links + ATT&CK links
4. Cell 4 – Ingest CVEs (NVD) with CWE bridge
5. Cell 5 – Verify graph health


---
## Cell 1 — Config, driver, constraints

In [8]:
from neo4j import GraphDatabase

URI  = "bolt://localhost:7687"
AUTH = ("neo4j", "KG_MITRE_V2.0")   # ← change if needed

driver = GraphDatabase.driver(URI, auth=AUTH)

CONSTRAINTS = [
    # Node uniqueness — prevents duplicates on re-runs
    "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Technique)  REQUIRE t.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (ta:Tactic)    REQUIRE ta.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (g:Group)      REQUIRE g.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Software)   REQUIRE s.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:CAPEC)      REQUIRE c.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (w:CWE)        REQUIRE w.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (v:CVE)        REQUIRE v.id IS UNIQUE",
    # Indexes for fast lookups
    "CREATE INDEX IF NOT EXISTS FOR (t:Technique)  ON (t.mitre_id)",
    "CREATE INDEX IF NOT EXISTS FOR (ta:Tactic)    ON (ta.mitre_id)",
]

with driver.session() as s:
    for stmt in CONSTRAINTS:
        s.run(stmt)

print("✅ Constraints and indexes ready.")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
✅ Constraints and indexes ready.


---
## Cell 2 — Ingest ATT&CK STIX bundle
Ingests: **Tactics**, **Techniques**, **Sub-techniques**, **Groups**, **Software (malware + tools)**  
Relationships: `BELONGS_TO`, `HAS_SUBTECHNIQUE`, `USES`

In [9]:
import json
from stix2 import MemoryStore, Filter

ATTACK_FILE = 'enterprise-attack.json' 

def _get(obj, key, default=None):
    """Unified getter: works for both stix2 objects and plain dicts."""
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)

def _id(obj):
    """Get the STIX id field from either a stix2 object or a dict."""
    return obj['id'] if isinstance(obj, dict) else obj.id

def _name(obj):
    return obj['name'] if isinstance(obj, dict) else obj.name

def mitre_id(obj):
    """Extract the MITRE ATT&CK external ID (e.g. T1059, TA0002, G0016)."""
    refs = _get(obj, 'external_references', [])
    for ref in refs:
        src = ref.get('source_name') if isinstance(ref, dict) else getattr(ref, 'source_name', '')
        if src == 'mitre-attack':
            return ref.get('external_id') if isinstance(ref, dict) else getattr(ref, 'external_id', None)
    return _id(obj)  # fallback to STIX id

def url(obj):
    refs = _get(obj, 'external_references', [])
    for ref in refs:
        src = ref.get('source_name') if isinstance(ref, dict) else getattr(ref, 'source_name', '')
        if src == 'mitre-attack':
            return ref.get('url', '') if isinstance(ref, dict) else getattr(ref, 'url', '')
    return ''

def run_full_attack_ingestion(file_path):
    print("Loading STIX bundle…")
    with open(file_path) as f:
        bundle = json.load(f)
    store = MemoryStore(stix_data=bundle)

    with driver.session() as s:
        # ── 1. TACTICS (x-mitre-tactic) ──────────────────────────────────────
        print("Ingesting Tactics…")
        tactics = store.query([Filter('type', '=', 'x-mitre-tactic')])
        for t in tactics:
            mid = mitre_id(t)
            s.run("""
                MERGE (ta:Tactic {stix_id: $stix_id})
                SET ta.mitre_id   = $mid,
                    ta.name       = $name,
                    ta.short_name = $short,
                    ta.description= $desc,
                    ta.url        = $url
            """, stix_id=_id(t), mid=mid, name=_name(t),
                 short=_get(t, 'x_mitre_shortname', ''),
                 desc=_get(t, 'description', ''), url=url(t))
        print(f"  → {len(tactics)} tactics")

        # ── 2. TECHNIQUES + SUB-TECHNIQUES (attack-pattern) ──────────────────
        print("Ingesting Techniques & Sub-techniques…")
        techniques = store.query([Filter('type', '=', 'attack-pattern')])
        # Filter out revoked/deprecated
        techniques = [t for t in techniques
                      if not t.get('revoked') and not t.get('x_mitre_deprecated')]
        for t in techniques:
            mid = mitre_id(t)
            is_sub = '.' in mid
            platforms = t.get('x_mitre_platforms', [])
            s.run("""
                MERGE (tech:Technique {stix_id: $stix_id})
                SET tech.mitre_id    = $mid,
                    tech.name        = $name,
                    tech.description = $desc,
                    tech.is_subtechnique = $is_sub,
                    tech.platforms   = $platforms,
                    tech.url         = $url
            """, stix_id=t.id, mid=mid, name=t.name,
                 desc=t.get('description', ''), is_sub=is_sub,
                 platforms=platforms, url=url(t))
        print(f"  → {len(techniques)} techniques/sub-techniques")

        # ── 3. GROUPS (intrusion-set) ─────────────────────────────────────────
        print("Ingesting Groups (Threat Actors)…")
        groups = store.query([Filter('type', '=', 'intrusion-set')])
        groups = [g for g in groups
                  if not g.get('revoked') and not g.get('x_mitre_deprecated')]
        for g in groups:
            mid = mitre_id(g)
            aliases = g.get('aliases', [])
            s.run("""
                MERGE (gr:Group {stix_id: $stix_id})
                SET gr.mitre_id   = $mid,
                    gr.name       = $name,
                    gr.aliases    = $aliases,
                    gr.description= $desc,
                    gr.url        = $url
            """, stix_id=g.id, mid=mid, name=g.name,
                 aliases=aliases, desc=g.get('description', ''), url=url(g))
        print(f"  → {len(groups)} groups")

        # ── 4. SOFTWARE (malware + tool) ──────────────────────────────────────
        print("Ingesting Software (Malware + Tools)…")
        software = (
            store.query([Filter('type', '=', 'malware')]) +
            store.query([Filter('type', '=', 'tool')])
        )
        software = [sw for sw in software
                    if not sw.get('revoked') and not sw.get('x_mitre_deprecated')]
        for sw in software:
            mid = mitre_id(sw)
            s.run("""
                MERGE (so:Software {stix_id: $stix_id})
                SET so.mitre_id   = $mid,
                    so.name       = $name,
                    so.type       = $sw_type,
                    so.description= $desc,
                    so.url        = $url
            """, stix_id=sw.id, mid=mid, name=sw.name,
                 sw_type=sw.type, desc=sw.get('description', ''), url=url(sw))
        print(f"  → {len(software)} software items")

        # ── 5. RELATIONSHIPS ──────────────────────────────────────────────────
        print("Building relationships…")
        rels = store.query([Filter('type', '=', 'relationship')])

        tactic_phase_map = {_get(t, 'x_mitre_shortname'): _id(t) for t in tactics}

        uses_count       = 0
        subtechnique_count = 0
        tactic_count     = 0

        # 5a. Technique → Tactic (via kill_chain_phases on each technique)
        for t in techniques:
            for phase in t.get('kill_chain_phases', []):
                short = phase.get('phase_name')
                ta_stix = tactic_phase_map.get(short)
                if ta_stix:
                    s.run("""
                        MATCH (tech:Technique {stix_id: $tech_id})
                        MATCH (ta:Tactic {stix_id: $ta_id})
                        MERGE (tech)-[:BELONGS_TO]->(ta)
                    """, tech_id=t.id, ta_id=ta_stix)
                    tactic_count += 1

        # 5b. Sub-technique → Parent technique
        for rel in rels:
            if rel.relationship_type == 'subtechnique-of':
                s.run("""
                    MATCH (parent:Technique {stix_id: $parent_id})
                    MATCH (sub:Technique   {stix_id: $sub_id})
                    MERGE (parent)-[:HAS_SUBTECHNIQUE]->(sub)
                """, parent_id=rel.target_ref, sub_id=rel.source_ref)
                subtechnique_count += 1

        # 5c. Group/Software USES Technique/Software
        for rel in rels:
            if rel.relationship_type == 'uses':
                s.run("""
                    MATCH (src {stix_id: $src})
                    MATCH (tgt {stix_id: $tgt})
                    MERGE (src)-[:USES]->(tgt)
                """, src=rel.source_ref, tgt=rel.target_ref)
                uses_count += 1

        print(f"  → {tactic_count} BELONGS_TO (Technique→Tactic)")
        print(f"  → {subtechnique_count} HAS_SUBTECHNIQUE")
        print(f"  → {uses_count} USES")

    print("\n✅ ATT&CK ingestion complete!")

run_full_attack_ingestion(ATTACK_FILE)

Loading STIX bundle…
Ingesting Tactics…
  → 14 tactics
Ingesting Techniques & Sub-techniques…
  → 691 techniques/sub-techniques
Ingesting Groups (Threat Actors)…
  → 172 groups
Ingesting Software (Malware + Tools)…
  → 784 software items
Building relationships…
  → 887 BELONGS_TO (Technique→Tactic)
  → 477 HAS_SUBTECHNIQUE
  → 17270 USES

✅ ATT&CK ingestion complete!


---
## Cell 3 — Ingest CAPEC
Ingests: **CAPEC nodes** with metadata  
Relationships: `CHILD_OF` (CAPEC hierarchy), `TARGETS_WEAKNESS` (CAPEC→CWE), `MAPS_TO_TECHNIQUE` (CAPEC→Technique)

**Fixes vs v1:**
- CAPEC nodes are now created with name + description + likelihood/severity
- CWE link uses `CWE_ID` attribute (not `.text`)
- CAPEC→CAPEC hierarchy is preserved
- Sub-technique IDs (e.g. T1574.010) try exact match first, then parent fallback

In [10]:
import xml.etree.ElementTree as ET

CAPEC_FILE = 'capec_v3.9.xml' 

def ingest_capec_full(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    ns   = {'c': 'http://capec.mitre.org/capec-3'}

    patterns = root.findall('.//c:Attack_Pattern', ns)
    print(f"Found {len(patterns)} CAPEC patterns")

    with driver.session() as s:
        # ── 1. Create all CAPEC nodes ─────────────────────────────────────────
        print("Creating CAPEC nodes…")
        for p in patterns:
            capec_id = f"CAPEC-{p.get('ID')}"
            name     = p.get('Name', '')
            abstr    = p.get('Abstraction', '')
            status   = p.get('Status', '')
            desc_el  = p.find('c:Description', ns)
            desc     = (desc_el.text or '').strip() if desc_el is not None else ''
            likely   = getattr(p.find('c:Likelihood_Of_Attack', ns), 'text', '') or ''
            severity = getattr(p.find('c:Typical_Severity', ns), 'text', '') or ''

            s.run("""
                MERGE (ca:CAPEC {id: $id})
                SET ca.name        = $name,
                    ca.abstraction = $abstr,
                    ca.status      = $status,
                    ca.description = $desc,
                    ca.likelihood  = $likely,
                    ca.severity    = $severity
            """, id=capec_id, name=name, abstr=abstr, status=status,
                 desc=desc, likely=likely, severity=severity)

        # ── 2. CAPEC → CWE  (TARGETS_WEAKNESS) ───────────────────────────────
        print("Linking CAPEC → CWE…")
        cwe_links = 0
        for p in patterns:
            capec_id = f"CAPEC-{p.get('ID')}"
            for rw in p.findall('.//c:Related_Weakness', ns):
                cwe_num = rw.get('CWE_ID')        # e.g. "276"
                if cwe_num:
                    cwe_id = f"CWE-{cwe_num}"
                    s.run("""
                        MATCH (ca:CAPEC {id: $capec_id})
                        MERGE (w:CWE {id: $cwe_id})
                        MERGE (ca)-[:TARGETS_WEAKNESS]->(w)
                    """, capec_id=capec_id, cwe_id=cwe_id)
                    cwe_links += 1
        print(f"  → {cwe_links} TARGETS_WEAKNESS relationships")

        # ── 3. CAPEC hierarchy  (CHILD_OF) ────────────────────────────────────
        print("Building CAPEC hierarchy…")
        hierarchy_links = 0
        for p in patterns:
            child_id = f"CAPEC-{p.get('ID')}"
            for r in p.findall('.//c:Related_Attack_Pattern', ns):
                if r.get('Nature') == 'ChildOf':
                    parent_id = f"CAPEC-{r.get('CAPEC_ID')}"
                    s.run("""
                        MATCH (child:CAPEC  {id: $child_id})
                        MATCH (parent:CAPEC {id: $parent_id})
                        MERGE (child)-[:CHILD_OF]->(parent)
                    """, child_id=child_id, parent_id=parent_id)
                    hierarchy_links += 1
        print(f"  → {hierarchy_links} CHILD_OF relationships")

        # ── 4. CAPEC → ATT&CK Technique  (MAPS_TO_TECHNIQUE) ─────────────────
        print("Linking CAPEC → ATT&CK Techniques…")
        atk_links = 0
        for p in patterns:
            capec_id = f"CAPEC-{p.get('ID')}"
            for m in p.findall('.//c:Taxonomy_Mapping', ns):
                if m.get('Taxonomy_Name') != 'ATTACK':
                    continue
                entry_el = m.find('c:Entry_ID', ns)
                if entry_el is None or not entry_el.text:
                    continue
                raw = entry_el.text.strip()
                tech_id   = raw if raw.startswith('T') else f"T{raw}"  # e.g. T1574.010
                parent_id = tech_id.split('.')[0]                        # e.g. T1574

                result = s.run("""
                    MATCH (ca:CAPEC {id: $capec_id})
                    MATCH (t:Technique)
                    WHERE t.mitre_id = $tech_id OR t.mitre_id = $parent_id
                    MERGE (ca)-[:MAPS_TO_TECHNIQUE]->(t)
                    RETURN count(*) AS cnt
                """, capec_id=capec_id, tech_id=tech_id, parent_id=parent_id)
                atk_links += result.single()['cnt']
        print(f"  → {atk_links} MAPS_TO_TECHNIQUE relationships")

    print("\n✅ CAPEC ingestion complete!")

ingest_capec_full(CAPEC_FILE)

Found 615 CAPEC patterns
Creating CAPEC nodes…
Linking CAPEC → CWE…
  → 1214 TARGETS_WEAKNESS relationships
Building CAPEC hierarchy…
  → 533 CHILD_OF relationships
Linking CAPEC → ATT&CK Techniques…
  → 432 MAPS_TO_TECHNIQUE relationships

✅ CAPEC ingestion complete!


---
## Cell 4 — Ingest CVEs from NVD
For each CVE: creates the CVE node, fetches its CWEs from NVD, and builds `CVE-[:CAUSE_WEAKNESS]->CWE`.

Supply your list of CVE IDs. Optionally add an NVD API key for higher rate limits (50 req/s vs 5 req/s).

filter settings ~
#### Security research — broad coverage
YEAR_FROM=2015, MIN_CVSS=0.0, LIMIT_PER_CWE=20

#### Threat intel / red team — recent high-impact only  
YEAR_FROM=2020, MIN_CVSS=7.0, LIMIT_PER_CWE=10

#### Executive dashboard — critical exploited vulns only
YEAR_FROM=2018, MIN_CVSS=9.0, LIMIT_PER_CWE=5
#### (CISA KEV already covers this well on its own)

In [13]:
import requests, nvdlib, time
from datetime import datetime

# ── Config ────────────────────────────────────────────────────────────────────
NVD_API_KEY  = None      # Get free key at https://nvd.nist.gov/developers/request-an-api-key
                         # Without key: 5 req/30s  |  With key: 50 req/30s
DELAY        = 6.5       # seconds between calls (safe without API key)
YEAR_FROM    = 2000      # ← only CVEs published from this year onward
YEAR_TO      = 2025      # ← only CVEs published up to this year (None = no upper limit)
MIN_CVSS     = 7.0       # ← only HIGH/CRITICAL (7.0+), set to 0.0 for all
LIMIT_PER_CWE = 10       # ← how many CVEs to pull per CWE from NVD

# ── Step 1: Build CVE ID list ─────────────────────────────────────────────────
def build_cve_list():
    all_cves = set()

    # Source A: CISA KEV — real-world exploited, highest signal
    print("=== Source A: CISA KEV ===")
    try:
        kev = requests.get(
            "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json",
            timeout=30
        ).json()
        kev_ids = set()
        for v in kev["vulnerabilities"]:
            year = int(v["cveID"].split("-")[1])
            if year >= YEAR_FROM and (YEAR_TO is None or year <= YEAR_TO):
                kev_ids.add(v["cveID"])
        all_cves.update(kev_ids)
        print(f"  +{len(kev_ids)} CVEs (filtered from {len(kev['vulnerabilities'])} total)")
    except Exception as e:
        print(f"  CISA KEV fetch failed: {e}")

    # Source B: NVD per CWE already in your graph
    print("\n=== Source B: NVD per graph CWE ===")
    driver_local = GraphDatabase.driver(URI, auth=AUTH)
    with driver_local.session() as s:
        cwe_ids = [r["cwe_id"] for r in s.run("MATCH (w:CWE) RETURN w.id AS cwe_id ORDER BY w.id")]
    driver_local.close()
    print(f"  Found {len(cwe_ids)} CWEs in graph")

    nvd_kwargs = {"limit": LIMIT_PER_CWE}
    if NVD_API_KEY:
        nvd_kwargs["key"] = NVD_API_KEY
    if YEAR_FROM:
        nvd_kwargs["pubStartDate"] = f"{YEAR_FROM}-01-01T00:00:00.000"
    if YEAR_TO:
        nvd_kwargs["pubEndDate"]   = f"{YEAR_TO}-12-31T23:59:59.999"

    for cwe_id in cwe_ids:
        try:
            results = nvdlib.searchCVE(cweId=cwe_id, **nvd_kwargs)
            # Apply CVSS filter
            filtered = []
            for r in results:
                try:
                    score = float(r.score[1])
                    if score >= MIN_CVSS:
                        filtered.append(r.id)
                except Exception:
                    filtered.append(r.id)  # keep if score unavailable
            all_cves.update(filtered)
            print(f"  {cwe_id} → +{len(filtered)} CVEs (of {len(results)} returned)")
            time.sleep(DELAY)
        except Exception as e:
            print(f"  {cwe_id} → error: {e}")
            time.sleep(DELAY * 2)  # back off on error

    print(f"\n✅ Total unique CVEs to ingest: {len(all_cves)}")
    return list(all_cves)

In [7]:
# ── Step 2: Fetch enrichment from NVD and write to graph ─────────────────────
def fetch_cve(cve_id):
    """Return (score, severity, published, cwes, nvd_notes) for one CVE."""
    kwargs = {"cveId": cve_id}
    if NVD_API_KEY:
        kwargs["key"] = NVD_API_KEY
    try:
        results = nvdlib.searchCVE(**kwargs)
    except Exception as e:
        print(f"  NVD error for {cve_id}: {e}")
        return None, None, None, [], []

    if not results:
        return None, None, None, [], []

    r = results[0]
    cwes, nvd_notes = [], []
    if hasattr(r, 'weaknesses'):
        for w in r.weaknesses:
            for d in w.description:
                (cwes if d.value.startswith('CWE-') else nvd_notes).append(d.value)

    score, severity, published = None, None, None
    try:
        score    = float(r.score[1])
        severity = r.score[2]          # LOW / MEDIUM / HIGH / CRITICAL
    except Exception:
        pass
    try:
        published = r.published[:10]   # "2021-12-10T..." → "2021-12-10"
    except Exception:
        pass

    return score, severity, published, cwes, nvd_notes

def ingest_cves(cve_ids):
    print(f"Ingesting {len(cve_ids)} CVEs…")
    total_links = 0
    orphaned    = 0

    with driver.session() as s:
        for i, cve_id in enumerate(cve_ids, 1):
            score, severity, published, cwes, nvd_notes = fetch_cve(cve_id)
            time.sleep(DELAY)

            # Create/update CVE node with full properties
            s.run("""
                MERGE (v:CVE {id: $id})
                SET v.cvss_score          = $score,
                    v.severity            = $severity,
                    v.published_date      = $published,
                    v.nvd_weakness_notes  = $notes
            """, id=cve_id, score=score, severity=severity,
                 published=published, notes=nvd_notes)

            for cwe in cwes:
                s.run("""
                    MATCH (v:CVE {id: $cve_id})
                    MERGE (w:CWE {id: $cwe_id})
                    MERGE (v)-[:CAUSE_WEAKNESS]->(w)
                """, cve_id=cve_id, cwe_id=cwe)
                total_links += 1

            if cwes:
                print(f"  [{i}/{len(cve_ids)}] {cve_id} ({severity}, {score}) → {cwes}")
            else:
                orphaned += 1
                print(f"  [{i}/{len(cve_ids)}] {cve_id} — no CWE (nvd_note: {nvd_notes})")

    print(f"\n✅ Done! {total_links} CAUSE_WEAKNESS links | {orphaned} CVEs with no CWE")

# ── Run ───────────────────────────────────────────────────────────────────────
CVE_IDS = build_cve_list()
ingest_cves(CVE_IDS)

=== Source A: CISA KEV ===
  +1541 CVEs (filtered from 1591 total)

=== Source B: NVD per graph CWE ===
  Found 336 CWEs in graph
  CWE-1007 → error: time data '2000-01-01T00:00:00.000' does not match format '%Y-%m-%d %H:%M'
  CWE-1021 → error: time data '2000-01-01T00:00:00.000' does not match format '%Y-%m-%d %H:%M'
  CWE-1037 → error: time data '2000-01-01T00:00:00.000' does not match format '%Y-%m-%d %H:%M'
  CWE-112 → error: time data '2000-01-01T00:00:00.000' does not match format '%Y-%m-%d %H:%M'
  CWE-113 → error: time data '2000-01-01T00:00:00.000' does not match format '%Y-%m-%d %H:%M'
  CWE-114 → error: time data '2000-01-01T00:00:00.000' does not match format '%Y-%m-%d %H:%M'
  CWE-116 → error: time data '2000-01-01T00:00:00.000' does not match format '%Y-%m-%d %H:%M'
  CWE-117 → error: time data '2000-01-01T00:00:00.000' does not match format '%Y-%m-%d %H:%M'
  CWE-118 → error: time data '2000-01-01T00:00:00.000' does not match format '%Y-%m-%d %H:%M'
  CWE-1188 → error: t

---
## Cell 5 — Graph health check

In [12]:
CHECKS = [
    # Node counts
    ("Tactics",      "MATCH (n:Tactic)    RETURN count(n) AS cnt"),
    ("Techniques",   "MATCH (n:Technique) WHERE NOT n.is_subtechnique RETURN count(n) AS cnt"),
    ("Sub-techniques","MATCH (n:Technique) WHERE n.is_subtechnique RETURN count(n) AS cnt"),
    ("Groups",       "MATCH (n:Group)     RETURN count(n) AS cnt"),
    ("Software",     "MATCH (n:Software)  RETURN count(n) AS cnt"),
    ("CAPEC",        "MATCH (n:CAPEC)     RETURN count(n) AS cnt"),
    ("CWE",          "MATCH (n:CWE)       RETURN count(n) AS cnt"),
    ("CVE",          "MATCH (n:CVE)       RETURN count(n) AS cnt"),
    # Relationship counts
    ("BELONGS_TO (Technique→Tactic)",   "MATCH ()-[r:BELONGS_TO]->() RETURN count(r) AS cnt"),
    ("HAS_SUBTECHNIQUE",                 "MATCH ()-[r:HAS_SUBTECHNIQUE]->() RETURN count(r) AS cnt"),
    ("USES (Group/SW → Technique)",      "MATCH ()-[r:USES]->() RETURN count(r) AS cnt"),
    ("MAPS_TO_TECHNIQUE (CAPEC→ATT&CK)","MATCH ()-[r:MAPS_TO_TECHNIQUE]->() RETURN count(r) AS cnt"),
    ("TARGETS_WEAKNESS (CAPEC→CWE)",     "MATCH ()-[r:TARGETS_WEAKNESS]->() RETURN count(r) AS cnt"),
    ("CHILD_OF (CAPEC hierarchy)",       "MATCH ()-[r:CHILD_OF]->() RETURN count(r) AS cnt"),
    ("CAUSE_WEAKNESS (CVE→CWE)",         "MATCH ()-[r:CAUSE_WEAKNESS]->() RETURN count(r) AS cnt"),
    # Orphan checks
    ("Orphaned CVE  (no CWE)",  "MATCH (v:CVE) WHERE NOT (v)-[:CAUSE_WEAKNESS]->() RETURN count(v) AS cnt"),
    ("Orphaned CWE  (no edges)","MATCH (w:CWE) WHERE NOT ()-[]-(w) RETURN count(w) AS cnt"),
    ("Orphaned CAPEC (no ATT&CK link)","MATCH (ca:CAPEC) WHERE NOT (ca)-[:MAPS_TO_TECHNIQUE]->() RETURN count(ca) AS cnt"),
    ("Truly isolated CAPEC (no ATT&CK AND no CWE)", "MATCH (ca:CAPEC) WHERE NOT (ca)-[:MAPS_TO_TECHNIQUE]->() AND NOT (ca)-[:TARGETS_WEAKNESS]->() RETURN count(ca) AS cnt"),
]

print(f"{'Metric':<42} {'Count':>8}")
print("-" * 52)
with driver.session() as s:
    for label, query in CHECKS:
        result = s.run(query).single()
        cnt = result['cnt'] if result else 0
        flag = "  ⚠️" if 'Orphaned' in label and cnt > 0 else ""
        print(f"{label:<42} {cnt:>8}{flag}")

print("\n✅ Health check complete.")

Metric                                        Count
----------------------------------------------------
Tactics                                          14
Techniques                                      216
Sub-techniques                                  475
Groups                                          172
Software                                        784
CAPEC                                           615
CWE                                             391
CVE                                            1541
BELONGS_TO (Technique→Tactic)                   887
HAS_SUBTECHNIQUE                                475
USES (Group/SW → Technique)                   16102
MAPS_TO_TECHNIQUE (CAPEC→ATT&CK)                398
TARGETS_WEAKNESS (CAPEC→CWE)                   1212
CHILD_OF (CAPEC hierarchy)                      533
CAUSE_WEAKNESS (CVE→CWE)                       1504
Orphaned CVE  (no CWE)                          209  ⚠️
Orphaned CWE  (no edges)                          0
Orphane

In [16]:
def coverage_report():
    queries = {
        "total_techniques": """
            MATCH (t:Technique) WHERE NOT t.is_subtechnique
            RETURN count(t) AS val
        """,
        "capec_covered": """
            MATCH (t:Technique) WHERE NOT t.is_subtechnique
              AND (:CAPEC)-[:MAPS_TO_TECHNIQUE]->(t)
            RETURN count(DISTINCT t) AS val
        """,
        "cwe_covered": """
            MATCH (t:Technique) WHERE NOT t.is_subtechnique
              AND (:CWE)<-[:TARGETS_WEAKNESS]-(:CAPEC)-[:MAPS_TO_TECHNIQUE]->(t)
            RETURN count(DISTINCT t) AS val
        """,
        "cve_full_chain": """
            MATCH (t:Technique) WHERE NOT t.is_subtechnique
              AND (:CVE)-[:CAUSE_WEAKNESS]->(:CWE)
                  <-[:TARGETS_WEAKNESS]-(:CAPEC)
                  -[:MAPS_TO_TECHNIQUE]->(t)
            RETURN count(DISTINCT t) AS val
        """,
        "groups_with_chain": """
            MATCH (g:Group)-[:USES]->(t:Technique)
                  <-[:MAPS_TO_TECHNIQUE]-(:CAPEC)
                  -[:TARGETS_WEAKNESS]->(:CWE)
                  <-[:CAUSE_WEAKNESS]-(:CVE)
            RETURN count(DISTINCT g) AS val
        """,
        "total_groups": """
            MATCH (g:Group) RETURN count(g) AS val
        """,
    }

    with driver.session() as s:
        results = {k: s.run(q).single()["val"] for k, q in queries.items()}

    total = results["total_techniques"]
    print(f"\n{'='*52}")
    print(f"  {'Layer':<30} {'Count':>6}  {'Coverage':>8}")
    print(f"{'='*52}")
    print(f"  {'Total techniques':<30} {total:>6}")
    print(f"  {'Covered by CAPEC':<30} {results['capec_covered']:>6}  {results['capec_covered']/total*100:>7.1f}%")
    print(f"  {'Covered by CWE→CAPEC':<30} {results['cwe_covered']:>6}  {results['cwe_covered']/total*100:>7.1f}%")
    print(f"  {'Full chain (CVE→CWE→CAPEC)':<30} {results['cve_full_chain']:>6}  {results['cve_full_chain']/total*100:>7.1f}%")
    print(f"{'='*52}")
    tg = results["total_groups"]
    gc = results["groups_with_chain"]
    print(f"  {'Groups with full CVE chain':<30} {gc:>6}/{tg:<6} {gc/tg*100:>6.1f}%")
    print(f"{'='*52}\n")

    # ── Per-tactic breakdown ──────────────────────────────────────────────────
    tactic_query = """
        MATCH (ta:Tactic)
        OPTIONAL MATCH (t:Technique)-[:BELONGS_TO]->(ta)
        WHERE NOT t.is_subtechnique
        WITH ta, count(DISTINCT t) AS total_t

        OPTIONAL MATCH (t2:Technique)-[:BELONGS_TO]->(ta)
        WHERE NOT t2.is_subtechnique
          AND (:CVE)-[:CAUSE_WEAKNESS]->(:CWE)
              <-[:TARGETS_WEAKNESS]-(:CAPEC)
              -[:MAPS_TO_TECHNIQUE]->(t2)
        WITH ta.name AS tactic, total_t,
             count(DISTINCT t2) AS cve_covered
        RETURN tactic, total_t, cve_covered,
               round(100.0 * cve_covered / total_t, 1) AS pct
        ORDER BY pct DESC
    """

    print(f"  {'Tactic':<35} {'Covered':>8} {'Total':>6} {'Pct':>6}")
    print(f"  {'-'*57}")
    with driver.session() as s:
        rows = s.run(tactic_query).data()
    for row in rows:
        bar_len = int(row["pct"] / 5)          # 1 char per 5%
        bar = "█" * bar_len + "░" * (20 - bar_len)
        print(f"  {row['tactic']:<35} {row['cve_covered']:>8} {row['total_t']:>6} {row['pct']:>5.1f}%  {bar}")
    print()

coverage_report()


  Layer                           Count  Coverage
  Total techniques                  216
  Covered by CAPEC                  102     47.2%
  Covered by CWE→CAPEC               95     44.0%
  Full chain (CVE→CWE→CAPEC)         85     39.4%
  Groups with full CVE chain        156/172      90.7%

  Tactic                               Covered  Total    Pct
  ---------------------------------------------------------
  Collection                                13     17  76.5%  ███████████████░░░░░
  Privilege Escalation                      10     14  71.4%  ██████████████░░░░░░
  Credential Access                         12     17  70.6%  ██████████████░░░░░░
  Lateral Movement                           6      9  66.7%  █████████████░░░░░░░
  Discovery                                 19     34  55.9%  ███████████░░░░░░░░░
  Persistence                               12     23  52.2%  ██████████░░░░░░░░░░
  Defense Evasion                           18     47  38.3%  ███████░░░░░░░░░░░░░
 

---
## Cell 6 — Example queries (read-only, run anytime)

These are starter Cypher queries to explore your graph.

In [9]:
SAMPLE_QUERIES = {

    "Full chain: CVE → CWE → CAPEC → Technique → Tactic": """
        MATCH (v:CVE)-[:CAUSE_WEAKNESS]->(w:CWE)
              <-[:TARGETS_WEAKNESS]-(ca:CAPEC)
              -[:MAPS_TO_TECHNIQUE]->(t:Technique)
              -[:BELONGS_TO]->(ta:Tactic)
        RETURN v.id, w.id, ca.id, ca.name, t.mitre_id, t.name, ta.name
        LIMIT 10
    """,

    "Groups using a specific technique (e.g. T1059 — Command Scripting)": """
        MATCH (g:Group)-[:USES]->(t:Technique {mitre_id: 'T1059'})
        RETURN g.name, g.mitre_id
        ORDER BY g.name
    """,

    "Top 10 most-used techniques across all groups": """
        MATCH (g:Group)-[:USES]->(t:Technique)
        RETURN t.mitre_id, t.name, count(g) AS group_count
        ORDER BY group_count DESC
        LIMIT 10
    """,

    "Software used by a group (e.g. APT29)": """
        MATCH (g:Group {name: 'APT29'})-[:USES]->(s:Software)
        RETURN s.name, s.type, s.mitre_id
    """,

    "Techniques under Credential Access tactic": """
        MATCH (t:Technique)-[:BELONGS_TO]->(ta:Tactic {short_name: 'credential-access'})
        WHERE NOT t.is_subtechnique
        RETURN t.mitre_id, t.name
        ORDER BY t.mitre_id
    """,

    "CAPEC→CWE→CVE for a given technique": """
        MATCH (ca:CAPEC)-[:MAPS_TO_TECHNIQUE]->(t:Technique {mitre_id: 'T1134'})
        OPTIONAL MATCH (ca)-[:TARGETS_WEAKNESS]->(w:CWE)
        OPTIONAL MATCH (v:CVE)-[:CAUSE_WEAKNESS]->(w)
        RETURN t.name, ca.id, ca.name, w.id, v.id
    """,
}

with driver.session() as s:
    for title, query in SAMPLE_QUERIES.items():
        print(f"\n{'='*60}")
        print(f"  {title}")
        print('='*60)
        rows = s.run(query).data()
        if rows:
            for row in rows:
                print(row)
        else:
            print("  (no results)")


  Full chain: CVE → CWE → CAPEC → Technique → Tactic
{'v.id': 'CVE-2023-2897', 'w.id': 'CWE-345', 'ca.id': 'CAPEC-141', 'ca.name': 'Cache Poisoning', 't.mitre_id': 'T1557', 't.name': 'Adversary-in-the-Middle', 'ta.name': 'Credential Access'}
{'v.id': 'CVE-2014-0364', 'w.id': 'CWE-345', 'ca.id': 'CAPEC-141', 'ca.name': 'Cache Poisoning', 't.mitre_id': 'T1557', 't.name': 'Adversary-in-the-Middle', 'ta.name': 'Credential Access'}
{'v.id': 'CVE-2017-9606', 'w.id': 'CWE-345', 'ca.id': 'CAPEC-141', 'ca.name': 'Cache Poisoning', 't.mitre_id': 'T1557', 't.name': 'Adversary-in-the-Middle', 'ta.name': 'Credential Access'}
{'v.id': 'CVE-2014-8165', 'w.id': 'CWE-345', 'ca.id': 'CAPEC-141', 'ca.name': 'Cache Poisoning', 't.mitre_id': 'T1557', 't.name': 'Adversary-in-the-Middle', 'ta.name': 'Credential Access'}
{'v.id': 'CVE-2014-4936', 'w.id': 'CWE-345', 'ca.id': 'CAPEC-141', 'ca.name': 'Cache Poisoning', 't.mitre_id': 'T1557', 't.name': 'Adversary-in-the-Middle', 'ta.name': 'Credential Access'}
{

In [10]:
MIGRATIONS = [
    # 1. Rename :Weakness → :CWE
    (
        "Promote :Weakness to :CWE",
        """
        MATCH (w:Weakness)
        MERGE  (c:CWE {id: w.id})
        SET    c += properties(w)
        WITH   w, c
        MATCH  (w)-[r]->(nb)
        MERGE  (c)-[:TARGETS_WEAKNESS]->(nb)
        DETACH DELETE w
        """
    ),
    # 2. EXPLOITED_BY → CAUSE_WEAKNESS
    (
        "Replace :EXPLOITED_BY with :CAUSE_WEAKNESS",
        """
        MATCH (v:CVE)-[r:EXPLOITED_BY]->(w:CWE)
        MERGE (v)-[:CAUSE_WEAKNESS]->(w)
        DELETE r
        """
    ),
    # 3. Remove catch-all :RELATED edges (replaced by typed relationships)
    (
        "Delete generic :RELATED relationships",
        "MATCH ()-[r:RELATED]->() DELETE r"
    ),
]

print("Running v1 → v2 migrations…")
with driver.session() as s:
    for label, cypher in MIGRATIONS:
        print(f"  {label}…")
        s.run(cypher)
print("\n✅ Migration complete!")

Running v1 → v2 migrations…
  Promote :Weakness to :CWE…
  Replace :EXPLOITED_BY with :CAUSE_WEAKNESS…
  Delete generic :RELATED relationships…

✅ Migration complete!
